# Módulo 0: Datos, calidad y EDA

**Objetivo:** entender la base `olist.duckdb`, detectar y resolver problemas de calidad, definir el
**período útil** y producir el EDA que justifica las decisiones de los módulos B y D.

**Reglas:** toda lectura con SQL (archivos en `sql/00_datos/`), conexión solo lectura, fechas como
parámetro `?`. Pandas solo a partir del resultado de la consulta.

**Responsable:** _(nombre)_

In [ ]:
# Configuración común: imports, conexión solo lectura y estilo de gráficos
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from olist import conectar, consulta, RANDOM_STATE
from olist import config
from olist.graficos import estilo, guardar

estilo()
np.random.seed(RANDOM_STATE)
con = conectar()  # read_only=True: la base cruda no se modifica

In [ ]:
con.sql("SHOW TABLES")

## 1. Granularidad y llaves

Verificar la unicidad de cada llave y la cardinalidad de cada relación (1:1, 1:N). Identificar qué
tablas multiplican filas al unirse y cómo evitarlo.

In [ ]:
llaves = consulta(con, "00_datos/llaves_unicidad.sql")
llaves["unica"] = llaves["filas"] == llaves["distintos"]
llaves

In [ ]:
consulta(con, "00_datos/cardinalidad_relaciones.sql")

**TODO:** interpretar. ¿Qué llaves no son únicas (p. ej. `geolocation`, `order_reviews.order_id`)?
¿Qué uniones multiplican filas y cómo se resuelven (agregar antes de unir)? Incluir el diagrama relacional en el informe.

## 2. Calidad de datos

Pendientes (crear un `.sql` por chequeo en `sql/00_datos/`):
- [x] Fechas faltantes e inconsistentes → `calidad_fechas.sql`
- [ ] Valores faltantes por tabla (productos sin categoría, dimensiones nulas, ...)
- [ ] Órdenes sin ítems
- [ ] Pagos que no cuadran con precio + flete
- [ ] Prefijos postales sin geolocalización (clientes y vendedores)
- [ ] Categorías sin traducción

In [ ]:
consulta(con, "00_datos/calidad_fechas.sql")

In [ ]:
# TODO: consulta(con, "00_datos/calidad_pagos.sql")

## 3. Identidad del cliente

**TODO:** explicar la diferencia entre `customer_id` (uno por orden) y `customer_unique_id`
(la persona) y sus consecuencias (p. ej. clientes recurrentes, frecuencia de compra).

In [ ]:
# TODO: consulta(con, "00_datos/identidad_cliente.sql")

## 4. Cobertura temporal y período útil

Hay meses al inicio y al final con muy pocas órdenes. Definir el rango a usar y **actualizar
`FECHA_INICIO_UTIL` y `FECHA_FIN_UTIL` en `src/olist/config.py`** (lo usan B y D).

In [ ]:
cobertura = consulta(con, "00_datos/cobertura_mensual.sql")
fig, ax = plt.subplots()
ax.bar(cobertura["mes"], cobertura["ordenes"], width=20)
ax.set_title("Órdenes por mes de compra")
ax.set_ylabel("órdenes")
guardar(fig, "cobertura_mensual", "00_datos")
cobertura

In [ ]:
print("Período útil provisional:", config.FECHA_INICIO_UTIL, "→", config.FECHA_FIN_UTIL)

**TODO:** justificar el período útil elegido.

## 5. EDA orientado a los módulos

- [x] Órdenes por día (insumo directo del Módulo B)
- [ ] Órdenes por semana
- [ ] Distribución de tiempos de entrega y del retraso
- [ ] Distribución geográfica de clientes y vendedores
- [ ] Comportamiento de compra por cliente

In [ ]:
diaria = consulta(con, "B_series/serie_diaria.sql", ["2016-09-01", "2018-11-01"])
fig, ax = plt.subplots()
ax.plot(diaria["fecha"], diaria["ordenes"], lw=0.8)
for f in (config.FECHA_INICIO_UTIL, config.FECHA_FIN_UTIL):
    ax.axvline(pd.Timestamp(f), color="red", ls="--")
ax.set_title("Órdenes por día (líneas rojas: período útil provisional)")
guardar(fig, "ordenes_diarias", "00_datos")

## Conclusiones del Módulo 0

**TODO:** decisiones de calidad y período útil que usan los demás módulos.

In [ ]:
con.close()